In [8]:
!pip install ultralytics --no-deps

^C


   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
    --------------------------------------- 0.8/44.0 MB 4.7 MB/s eta 0:00:10
   - -------------------------------------- 1.8/44.0 MB 5.5 MB/s eta 0:00:08
   --- ------------------------------------ 3.4/44.0 MB 6.0 MB/s eta 0:00:07
   ---- ----------------------------------- 5.0/44.0 MB 6.2 MB/s eta 0:00:07
   ----- ---------------------------------- 5.8/44.0 MB 6.2 MB/s eta 0:00:07
   ------ --------------------------------- 7.3/44.0 MB 6.2 MB/s eta 0:00:06
   -------- ------------------------------- 8.9/44.0 MB 6.3 MB/s eta 0:00:06
   --------- ------------------------------ 10.2/44.0 MB 6.2 MB/s eta 0:00:06
   ---------- ----------------------------- 11.5/44.0 MB 6.2 MB/s eta 0:00:06
   ----------- ---------------------------- 13.1/44.0 MB 6.3 MB/s eta 0:00:05
   ------------- -------------------------- 14.4/44.0 MB 6.4 MB/s eta 0:00:05
   ------------- -------------------------- 15.2/44.0 MB 6.4 MB/s eta 0:00:05
 

In [9]:
!pip install opencv-python

In [ ]:
import os
import shutil
import yaml
import gc
from ultralytics import YOLO, settings
import torch
import numpy as np

# Riproducibilità
torch.manual_seed(0)
np.random.seed(0)
torch.backends.cudnn.deterministic = True 
torch.backends.cudnn.benchmark = False

# Impostazioni generali
PROJECT = "pothole-detector-NatureSR"
RANDOM_SEED = 0
IOU_THRESHOLD = 0.7
CONF_THRESHOLD = 0.25
INPUT_SIZE = 640
PATIENCE = 10
EPOCHS = 100

# Dataset paths
data_dir = "./data"

# Configurazione parametri (run gj4071db)
config = {
    'batch': 8,
    'optimizer': "RAdam",
    'lr0': 0.007023087386876883,
    'lrf': 0.02559244954708425,
    'weight_decay': 0.005034138135400871,
    'momentum': 0.09079056013311568,
    'dropout': 0.11321865578015432
}

In [21]:
def create_yaml():
    yaml_path = "data.yaml"
    data = {
        'path': os.path.abspath(data_dir),
        'train': 'images/train',
        'val': 'images/val',
        'nc': 3,
        'names': ['pothole', 'crack', 'manhole']
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f)
    print(f"✅ File YAML created: {yaml_path}")
    return yaml_path

In [22]:
import os
import shutil
import random
import yaml
import gc
import torch
import numpy as np
from ultralytics import YOLO, settings

# ============================================================
# 🔧 ตรวจสอบอุปกรณ์ (GPU/CPU) อัตโนมัติ
# ============================================================
def get_device():
    """ตรวจสอบว่ามี GPU (CUDA) ใช้งานได้หรือไม่ ถ้าไม่มีให้ใช้ CPU แทน"""
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✅ พบ GPU: {gpu_name} -> ใช้ device='0'")
        return 0
    else:
        print("⚠️ ไม่พบ GPU (CUDA) -> จะใช้ CPU แทน (การเทรนจะช้ากว่ามาก)")
        return "cpu"

DEVICE = get_device()

# ============================================================
# Riproducibilità
# ============================================================
RANDOM_SEED = 0
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# เปิด cudnn optimization เฉพาะตอนมี GPU เท่านั้น
if DEVICE != "cpu":
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================
# Impostazioni generali
# ============================================================
PROJECT = "pothole-detector-NatureSR"
IOU_THRESHOLD = 0.7
CONF_THRESHOLD = 0.25
INPUT_SIZE = 640
PATIENCE = 10
EPOCHS = 5 

# ปรับ batch size และ workers อัตโนมัติตามอุปกรณ์
BATCH_SIZE = 8 if DEVICE != "cpu" else 4      # CPU ใช้ batch เล็กลงกันแรม/ซีพียูโหลดหนัก
WORKERS = 4 if DEVICE != "cpu" else 0         # workers>0 บน Windows+CPU มักมีปัญหา multiprocessing

# ============================================================
# Dataset paths (ใช้ relative path แทน path แบบ Kaggle)
# ============================================================
data_dir = os.path.join(os.getcwd(), "data")

# ============================================================
# Configurazione parametri (run gj4071db)
# ============================================================
config = {
    'batch': BATCH_SIZE,
    'optimizer': "RAdam",
    'lr0': 0.007023087386876883,
    'lrf': 0.02559244954708425,
    'weight_decay': 0.005034138135400871,
    'momentum': 0.09079056013311568,
    'dropout': 0.11321865578015432
}

# ============================================================
# แบ่งข้อมูล Train/Val
# ============================================================
def split_train_val(data_dir, images_folder="images", labels_folder="labels-YOLO",
                     val_ratio=0.2, seed=0):
    random.seed(seed)
    images_path = os.path.join(data_dir, images_folder)
    labels_path = os.path.join(data_dir, labels_folder)

    valid_ext = ('.jpg', '.jpeg', '.png', '.bmp')
    image_files = [f for f in os.listdir(images_path) if f.lower().endswith(valid_ext)]

    print(f"📊 จำนวนรูปภาพทั้งหมด: {len(image_files)}")

    missing_labels = []
    for img_file in image_files:
        label_file = os.path.splitext(img_file)[0] + ".txt"
        if not os.path.exists(os.path.join(labels_path, label_file)):
            missing_labels.append(img_file)

    if missing_labels:
        print(f"⚠️ พบรูปภาพ {len(missing_labels)} ไฟล์ที่ไม่มี label คู่กัน")

    random.shuffle(image_files)
    val_size = int(len(image_files) * val_ratio)
    val_files = image_files[:val_size]
    train_files = image_files[val_size:]

    print(f"✅ Train: {len(train_files)} รูป | Val: {len(val_files)} รูป")

    for split in ['train', 'val']:
        os.makedirs(os.path.join(data_dir, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(data_dir, 'labels', split), exist_ok=True)

    def move_files(file_list, split_name):
        for img_file in file_list:
            label_file = os.path.splitext(img_file)[0] + ".txt"
            src_img = os.path.join(images_path, img_file)
            dst_img = os.path.join(data_dir, 'images', split_name, img_file)
            src_label = os.path.join(labels_path, label_file)
            dst_label = os.path.join(data_dir, 'labels', split_name, label_file)
            shutil.copy2(src_img, dst_img)
            if os.path.exists(src_label):
                shutil.copy2(src_label, dst_label)

    move_files(train_files, 'train')
    move_files(val_files, 'val')
    print("🎉 แบ่งข้อมูลเสร็จสมบูรณ์!")

# ============================================================
# สร้างไฟล์ data.yaml
# ============================================================
def create_yaml():
    """สร้างไฟล์ YAML ที่ชี้ไปยัง train/val แยกโฟลเดอร์"""
    yaml_path = "data.yaml"
    data = {
        'path': data_dir,
        'train': 'images/train',
        'val': 'images/val',
        'nc': 3,
        'names': ['pothole', 'crack', 'manhole']
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f)
    print(f"✅ File YAML created: {yaml_path}")
    return yaml_path

# ============================================================
# Training del modello YOLO
# ============================================================
def train_model(data_yaml_path, config, device=DEVICE):
    print(f"\n🔧 Avvio training con configurazione:")
    for key, value in config.items():
        print(f"   - {key}: {value}")

    # Inizializza modello
    model = YOLO("yolo11n.pt")

    # Training
    print("\n🚀 Training in corso...")
    try:
        results = model.train(
            data=data_yaml_path,
            epochs=EPOCHS,
            batch=config['batch'],
            imgsz=INPUT_SIZE,
            optimizer=config['optimizer'],
            lr0=config['lr0'],
            lrf=config['lrf'],
            weight_decay=config['weight_decay'],
            momentum=config['momentum'],
            dropout=config['dropout'],
            patience=PATIENCE,
            device=device,
            amp=True if device != "cpu" else False,   # amp (mixed precision) ใช้ได้เฉพาะ GPU
            seed=RANDOM_SEED,
            deterministic=True,
            project=PROJECT,
            name="training",
            exist_ok=True,
            val=True,
            save=True,
            plots=True,
            verbose=True,
            workers=WORKERS,
        )
    except torch.cuda.OutOfMemoryError:
        print("❌ GPU memory ไม่พอ! ลองลด batch size หรือ imgsz แล้วรันใหม่")
        raise
    except RuntimeError as e:
        print(f"❌ เกิดข้อผิดพลาดระหว่างเทรน: {e}")
        raise

    # Validation
    print("\n📊 Validazione finale...")
    val_results = model.val(
        data=data_yaml_path,
        iou=IOU_THRESHOLD,
        conf=CONF_THRESHOLD,
        device=device,
    )

    metrics = val_results.box
    class_names = ['pothole', 'crack', 'manhole']

    # Stampa risultati per classe (เช็ค index ให้ไม่เกินจำนวน class ที่ตรวจพบจริง)
    print("\n✅ Risultati per classe:")
    print("-" * 70)
    print(f"{'Classe':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'mAP50':<12}")
    print("-" * 70)

    n_detected_classes = len(metrics.ap50) if hasattr(metrics, 'ap50') else 0

    for i, name in enumerate(class_names):
        if i < n_detected_classes:
            precision = float(metrics.p[i])
            recall = float(metrics.r[i])
            f1 = float(metrics.f1[i])
            ap50 = float(metrics.ap50[i])
            print(f"{name:<12} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {ap50:<12.4f}")
        else:
            print(f"{name:<12} {'N/A (ไม่พบข้อมูล class นี้ใน validation set)':<40}")

    # Stampa medie
    print("-" * 70)
    print(f"{'MEDIA':<12} {float(metrics.mp):<12.4f} {float(metrics.mr):<12.4f} "
          f"{float(metrics.f1.mean()):<12.4f} {float(metrics.map50):<12.4f}")
    print("-" * 70)

    print(f"\n✅ Training completato!")
    print(f"   - mAP50: {float(metrics.map50):.4f}")
    print(f"   - mAP50-95: {float(metrics.map):.4f}")
    print(f"   - Fitness: {float(metrics.fitness()):.4f}")

    # Exporting ONNX
    try:
        model.export(format="onnx")
        print("✅ Export ONNX สำเร็จ")
    except Exception as e:
        print(f"⚠️ Export ONNX ไม่สำเร็จ: {e}")

    # Cleanup
    del model
    if device != "cpu":
        torch.cuda.empty_cache()
    gc.collect()

    return val_results


# ============================================================
# 🚀 Entry point (จำเป็นบน Windows เพื่อป้องกันปัญหา multiprocessing)
# ============================================================
if __name__ == "__main__":
    split_train_val(data_dir)
    yaml_path = create_yaml()
    train_model(data_yaml_path=yaml_path, config=config)

⚠️ ไม่พบ GPU (CUDA) -> จะใช้ CPU แทน (การเทรนจะช้ากว่ามาก)
📊 จำนวนรูปภาพทั้งหมด: 2009
✅ Train: 1608 รูป | Val: 401 รูป
🎉 แบ่งข้อมูลเสร็จสมบูรณ์!
✅ File YAML created: data.yaml

🔧 Avvio training con configurazione:
   - batch: 4
   - optimizer: RAdam
   - lr0: 0.007023087386876883
   - lrf: 0.02559244954708425
   - weight_decay: 0.005034138135400871
   - momentum: 0.09079056013311568
   - dropout: 0.11321865578015432

🚀 Training in corso...
Ultralytics 8.4.154  Python-3.14.6 torch-2.14.0+cpu CPU (AMD Ryzen 3 3250U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dn

: 

In [6]:
# Crea YAML
data_yaml_path = create_yaml()

# Training
results = train_model(data_yaml_path, config)

print("\n🏁 Processo completato!")

✅ File YAML creato: data.yaml

🚀 Avvio training con configurazione:
   - batch: 8
   - optimizer: RAdam
   - lr0: 0.007023087386876883
   - lrf: 0.02559244954708425
   - weight_decay: 0.005034138135400871
   - momentum: 0.09079056013311568
   - dropout: 0.11321865578015432

⏳ Training in corso...
Ultralytics 8.3.239 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.11321865578015432, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, ko

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       2009       4737      0.579       0.54      0.551      0.267
               pothole        795       1261      0.537      0.452      0.454      0.199
                 crack       1375       2519      0.522      0.357      0.392      0.168
               manhole        759        957      0.678       0.81      0.808      0.433
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /kaggle/working/pothole-detector-NatureSR/training

📊 Validazione finale...
Ultralytics 8.3.239 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 144.7±76.1 MB/s, size: 85.8 KB)
val: Scanning /kaggle/input/road-damage-dataset-potholes-cracks-and-manholes/data/labels... 2009 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2009/2009 833.4it/s 2.4s
WARNING ⚠️ val: Cache directory /kaggle/input/r

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       2009       4737      0.649      0.484      0.572      0.311
               pothole        795       1261      0.611      0.392      0.474      0.244
                 crack       1375       2519      0.593      0.275       0.43      0.219
               manhole        759        957      0.744      0.785      0.812       0.47
Speed: 0.5ms preprocess, 1.5ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /kaggle/working/runs/detect/val

📈 Risultati per classe:
----------------------------------------------------------------------
Classe       Precision    Recall       F1           mAP50       
----------------------------------------------------------------------
pothole      0.6106       0.3918       0.4773       0.4744      
crack        0.5928       0.2751       0.3758       0.4297      
manhole      0.7436       0.7847       0.7636       0.8118      
----------------------------------------------------------------------
MEDIA        0.